# WHOMOLLM Project
## Geo context prompt (Version 1) 24.Mar.2026 based on the Deepseek-llm-7b model feature: age group and household size, gender, household income level
1. Raw data
2. stay segmentation
3. reverse geocoding
4. top 5 POIs aggregation
5. semantic stay table
6. LLM prompt (daily narrative/ activity inference/ other surplus info)

## 1. Test GPU, Deepseek model 

In [2]:
import os
import json
import re
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
# ===============================
# GPU CHECK
# ===============================

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "deepseek-ai/deepseek-llm-7b-chat"

# ✅ 4bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,   
    trust_remote_code=True,
    use_safetensors=True
)

prompt = "Explain how people choose routes in a city."

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights: 100%|██████████| 273/273 [02:27<00:00,  1.85it/s] 
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Explainhowpeoplechooseroutesinacity.Urbana,ĠILĠisĠaĠcollegeĠtown.ĠTheĠUniversityĠofĠIllinoisĠisĠtheĠlargestĠuniversityĠinĠtheĠstate,ĠandĠitĠisĠlocatedĠinĠUrbana.ĠTheĠcityĠisĠalsoĠhomeĠtoĠParklandĠCollege,ĠaĠcommunityĠcollege.ĠManyĠpeopleĠwhoĠliveĠinĠUrbanaĠareĠstudentsĠorĠfacultyĠmembersĠatĠtheĠUniversityĠofĠIllinois,ĠorĠtheyĠworkĠatĠParklandĠCollege.ĠInĠaddition,ĠthereĠareĠmanyĠbusinessesĠinĠUrbanaĠthatĠcaterĠtoĠtheĠneedsĠofĠtheĠuniversityĠandĠitsĠstudents,ĠsuchĠasĠbookstores,Ġrestaurants,ĠandĠcoffeeĠshops.ĠTheĠcityĠalsoĠhasĠaĠpublicĠtransportationĠsystem,ĠwhichĠincludesĠbusesĠandĠaĠtrain,ĠthatĠhelpsĠpeopleĠgetĠaroundĠtheĠcity.Ġ(Source:ĠĊHowĠdoesĠtheĠcityĠofĠUrbana,ĠILĠmakeĠdecisionsĠaboutĠtransportation?ĊTheĠcityĠofĠUrbana,ĠILĠmakesĠdecisionsĠaboutĠtransportationĠthroughĠaĠcombinationĠofĠplanning,Ġpolicy-making,ĠandĠpublicĠinput.ĠTheĠcity'sĠtransportationĠdepartmentĠisĠresponsibleĠforĠdevelopingĠandĠimplementingĠtransportationĠplansĠandĠpoliciesĠthatĠaddressĠtheĠneedsĠofĠtheĠcommunit

## 2. Generate prompts ( 1st version with just geo context information and behaviour info, predict household size and age group ... for all users used different model, Deepseek 7B)
Version control:
This is the basic prompt based on geocontext and behaviour info, version 1.

In [5]:
#   ---------------------------
# Version 1 (24 Mar 2026) prompt with geo context to predict features
#   ---------------------------

import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb

# ---------------------------
# Config
# ---------------------------
RANDOM_SEED = 42
N_USERS     = 5
MAX_EVENTS  = 300
HOUR_BIN    = 1
PREC        = int(os.getenv("COORD_PREC", "4"))


DATA_SP     = Path("/data/baliu/python_code/data/sp_all copy.csv")
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v4_features_04Mar2026_age_hhsize.json")

# -------------------------------------------   
# Load (strings) + base cleaning
# -------------------------------------------
sp = pd.read_csv(DATA_SP, dtype=str, engine="python", on_bad_lines="skip")
print("sp shape:", sp.shape)

# Datetimes
sp["started_at"]  = pd.to_datetime(sp["started_at"], errors="coerce", utc=True)
sp["finished_at"] = pd.to_datetime(sp.get("finished_at"), errors="coerce", utc=True)

# Drop unusable
sp = sp.dropna(subset=["user_id", "started_at", "location_id"]).copy()
sp["user_id"]     = sp["user_id"].astype(str)
sp["location_id"] = sp["location_id"].astype(str)

#sp["duration_min"] = ((sp["finished_at"] - sp["started_at"]).dt.total_seconds() / 60.0).astype(float)
print(sp["duration"].head())
print(sp["duration"].dtype)

# Force numeric
sp["act_duration"] = pd.to_numeric(
    sp["act_duration"],
    errors="coerce"
)

# change it to hours
sp["act_duration_h"] = (sp["act_duration"] / 60.0).round(1)
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)

# length_km: convert to numeric, coerce errors to NaN, then divide by 1000
sp["length_km"] = pd.to_numeric(sp["length"], errors="coerce") / 1000.0
sp["length_km"] = sp["length_km"].round(1)

# Time features
sp["date"]     = sp["started_at"].dt.date
sp["dow"]      = sp["started_at"].dt.dayofweek.astype(int)        # 0=Mon
sp["hour_bin"] = (sp["started_at"].dt.hour // HOUR_BIN * HOUR_BIN).astype(int)

# ---------------------------
# Format from wkt point -> lon/lat (rounded)
# --------------------------------
WKT_POINT_RE = re.compile(
    r"POINT\s*\(\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*\)",
    re.I,
)

def extract_lonlat_from_wkt(s: str):
    m = WKT_POINT_RE.search(str(s))
    if not m: return (np.nan, np.nan)
    return float(m.group(1)), float(m.group(2))

sp["geometry"] = sp.get("geometry", pd.Series(index=sp.index)).astype(str)
lonlat = sp["geometry"].apply(extract_lonlat_from_wkt)
sp[["lon","lat"]] = pd.DataFrame(lonlat.tolist(), index=sp.index)
sp["lon"] = pd.to_numeric(sp["lon"], errors="coerce").round(PREC)
sp["lat"] = pd.to_numeric(sp["lat"], errors="coerce").round(PREC)

# Lightweight tags
sp["geometry_type"] = np.where(sp["lon"].notna() & sp["lat"].notna(), "point", "-")
if "mode" not in sp.columns: sp["mode"] = "-"

sp shape: (1152987, 10)
0    1093.0
1      39.0
2      75.0
3     164.0
4      36.0
Name: duration, dtype: str
str
0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64


In [6]:
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)
print(sp["length_km"].head())
print(sp["length_km"].dtype)

0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64
0     0.0
1    11.6
2     2.1
3     4.8
4    12.6
Name: length_km, dtype: float64
float64


In [7]:
print ("sp after cleaning:", sp.shape)
print (sp.head(5))

sp after cleaning: (862871, 18)
  id user_id                       started_at  \
0  0   AAGAF 2019-10-09 11:30:34.141000+00:00   
1  1   AAGAF 2019-10-10 06:14:49.141999+00:00   
2  2   AAGAF 2019-10-10 07:03:24.426000+00:00   
3  3   AAGAF 2019-10-10 11:10:24.605999+00:00   
4  4   AAGAF 2019-10-10 14:30:45.187999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-10 05:43:17.674999+00:00           0   Car                 0.0   
1 2019-10-10 06:53:54.841000+00:00           1   Car  11615.408548376423   
2 2019-10-10 08:18:20.864000+00:00           2   Car  2104.8558581266784   
3 2019-10-10 13:54:34.799339+00:00           2  Walk   4847.706521391238   
4 2019-10-10 15:07:07.127239+00:00           3   Car  12621.909934521313   

                                       geometry duration  act_duration  \
0  POINT (7.565219245342489 47.545616372908086)   1093.0        1093.0   
1  POINT (7.5637597959179645 47.54794767256367)     39.0          71

In [8]:
# ---------------------------
# Sample 1 week per user
# ---------------------------
def sample_one_week_per_user(sp, seed=42):
    rng = np.random.default_rng(seed)
    sp = sp.copy()
    sp["date"] = pd.to_datetime(sp["date"])

    out = []
    for uid, df_u in sp.groupby("user_id"):
        days = (
            df_u[["date"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(days) < 7:
            continue
        days["delta"] = days["date"].diff().dt.days.fillna(1).astype(int)
        days["block"] = (days["delta"] >1).cumsum()

        valid_blocks = (
            days.groupby("block")
            .filter(lambda x: len(x) >=7)
        )

        if valid_blocks.empty:
            continue
        candidate_starts = []
        for _, g in valid_blocks.groupby("block"):
            block_dates = g["date"].sort_values().reset_index(drop=True)
            for i in range(len(block_dates) - 6):
                candidate_starts.append(g.iloc[i]["date"])

        start_date = rng.choice(candidate_starts)
        week_dates = pd.date_range(start=start_date, periods=7, freq= 'D')

        out.append(
            df_u[df_u["date"].isin(week_dates)]

        )
       
    return pd.concat(out, ignore_index=True)

sp_sampled2 = sample_one_week_per_user(sp, seed=RANDOM_SEED)
print("sp_sampled2 shape:", sp_sampled2.shape)
print(sp_sampled2.head(2))

# ---------------------------
# save sampled data
# ---------------------------
SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_1week_04Mar2026_qwen.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)
print("Saved sp_sampled2_1week_04Mar2026 to", SP_SAMPLED2_OUT)  

sp_sampled2 shape: (60738, 18)
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   

                       finished_at location_id  mode             length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus  3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram  8232.492568529495   

                                       geometry duration  act_duration  \
0  POINT (7.564359990583688 47.546459921667946)   1097.0        6456.0   
1   POINT (7.603269084785703 47.58100341208616)    116.0         275.0   

   act_duration_h  length_km       date  dow  hour_bin     lon      lat  \
0           107.6        3.8 2019-10-22    1         9  7.5644  47.5465   
1             4.6        8.2 2019-10-23    2         6  7.6033  47.5810   

  geometry_type  
0         point  
1         point  
Saved sp_sampled2_1week_04Mar2026 to /data/baliu/python_code/data/sp_sampled2_1week_04Mar

#### Load location name and POIs top 5

In [ ]:
# 1. Load toponym data from shepefiles
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb
import geopandas as gpd
import fiona
# read parquet file
import pyarrow as pa
import pyarrow.parquet as pq
try:
    pa.unregister_extension_type("pandas.period")
except:
    pass
 
CACHE_DIR   = Path("/data/baliu/python_code/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NOM_CACHE_PATH = CACHE_DIR / "nominatim_cache.parquet"
#nom = pd.read_parquet(NOM_CACHE_PATH)

table = pq.read_table(NOM_CACHE_PATH)
df = table.to_pandas()

print("Nominatim cache loaded:", df.shape)
print("Nominatim columns:", df.columns.tolist())
df.head(30)
groupby_category = df.groupby('city').size()
print(groupby_category) 

ArrowKeyError: A type extension with name pandas.interval already defined

In [20]:
# left join sp with nominatim data to get city names
def attach_nominatim(sp_sampled2, nom):
    cols = ["lon", "lat", "road", "neighbourhood", "city", "postcode"]

    for c in ["lon", "lat"]:
        sp_sampled2[c] = sp_sampled2[c].astype(float).round(4)
        nom[c] = nom[c].astype(float).round(4)

    nom_sel = nom[cols].drop_duplicates(subset=["lon", "lat"])

    return sp_sampled2.merge(nom_sel, on=["lon", "lat"], how="left")

def clean_nominatim_fields(df):
    cols = ["road", "neighbourhood", "city", "postcode"]

    for c in cols:
        df[c] = (
            df[c]
            .replace(
                to_replace=[None, "None", "none", "", "-", "nan"],
                value=np.nan
            )
        )
    return df


sp_sampled2 = attach_nominatim(sp_sampled2, df)
sp_sampled2 = clean_nominatim_fields(sp_sampled2)
print("sp_sampled2 after attaching nominatim:", sp_sampled2.shape)
print("sp_sampled2 columns:", sp_sampled2.columns.tolist())   
print(sp_sampled2.head(10))

sp_sampled2 after attaching nominatim: (60738, 22)
sp_sampled2 columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode']
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   
5  28   AAGAF 2019-10-24 06:05:44.141999+00:00   
6  29   AAGAF 2019-10-24 07:22:44.510999+00:00   
7  30   AAGAF 2019-10-24 09:55:47.243000+00:00   
8  31   AAGAF 2019-10-24 13:05:05.668999+00:00   
9  32   AAGAF 2019-10-24 15:03:21.792999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10 

#### Build token with geocontext info

In [21]:
# 2. Load POI data from shapefiles
import geopandas as gpd
from pathlib import Path
import pandas as pd
import fiona
from shapely.geometry import Point, LineString, Polygon

POI_PATH = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")

poi = gpd.read_file(POI_PATH)
poi.head()

,id,code,name,category,geometry
0,0,3100,Temple de Saint-Jean,Civic,POINT (2498816.948 1117839.539)
1,1,3100,Kapelle Oberes Heiliges Kreuz,Civic,POINT (2691857.357 1192677.631)
2,2,3102,Kirche St. Johannes,Civic,POINT (2599659.594 1202970.041)
3,3,3102,Paroisse catholique,Civic,POINT (2501879.911 1126418.086)
4,4,3300,Mosquée du Petit-Saconnex,Civic,POINT (2498411.634 1119984.893)


In [22]:
poi = poi.copy()
print(poi["category"].value_counts())
print(f"poi crs: {poi.crs}")

# print("Category counts:", category_counts)

# new category counts without buildings and unknown
# category
# Others            158579
# Unknown           114894
# Entertainment     106734
# Shopping           54634
# Residential        45556
# Transportation     40997
# Services            9525
# Schools             2904
# Civic                780
# Name: count, dtype: int64
# poi crs: EPSG:2056

# category
# Unknown           2892609
# Others             158579
# Entertainment      106734
# Shopping            54634
# Residential         45556
# Transportation      40997
# Services             9525
# Schools              2904
# Civic                 780
# Name: count, dtype: int64
# poi crs: EPSG:2056



category
Others            158579
Unknown           114894
Entertainment     106734
Shopping           54634
Residential        45556
Transportation     40997
Services            9525
Schools             2904
Civic                780
Name: count, dtype: int64
poi crs: EPSG:2056


In [23]:
def load_pois_once():
    global poi
    if poi is None:
        print("🔹 Loading POIs into memory...")
        poi = (
            gpd.read_file(POI_PATH)  # metric CRS for distance
        )
        poi = poi.set_crs(epsg=2056, allow_override=True)
    return poi
# ---------------------------
# POI context function backup for sp_sampled1, 
#   with option to return details of nearest POIs
# ---------------------------
print(poi.head(2))

   id  code                           name category  \
0   0  3100           Temple de Saint-Jean    Civic   
1   1  3100  Kapelle Oberes Heiliges Kreuz    Civic   

                          geometry  
0  POINT (2498816.948 1117839.539)  
1  POINT (2691857.357 1192677.631)  


In [24]:
loc = gpd.GeoDataFrame(
    sp_sampled2,
    geometry=gpd.points_from_xy(sp_sampled2.lon, sp_sampled2.lat),
    crs="EPSG:4326"
).to_crs(2056)

##### update 
Top 5 buffer
clean text part () function
skip poi if both name and category unknown
";".join(out) string return
debug prints after merge

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from pathlib import Path

def clean_addr_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s

dow_names = {
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
    0: "Sunday" 
}

hour_bin_labels = {
    0: "midnight", #00:00-03:59
    1: "working hours in the early morning", #04:00-8:59
    2: "morning", #09:00-11:59
    3: "afternoon", #12:00-16:59
    4: "evening", #17:00-20:59
    5: "night", #21:00-23:59
}

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360
    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

def clean_text_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s

def get_poi_context_for_prompt(sp_sampled, poi, top_k=5):
    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled["lon"].astype(float),
            sp_sampled["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on POIs ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 5)  # large buffer to survive category filtering

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name":        poi.iloc[j]["name"],
                "category":    poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]

    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[["location_id", "name", "category", "addr_poi_dist_km", "direction"]]

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        name     = clean_text_part(r.get("name"))
        category = clean_text_part(r.get("category"))

        # skip POI if both name and category are unknown
        if name is None and category is None:
            continue

        label = " ".join(filter(None, [category, name]))
        out.append(
            f"{r['addr_poi_dist_km']} km to the {r['direction'].lower()}: {label}"
        )
    return "; ".join(out)  # string, CSV-safe and prompt-ready

# ---- build and merge ----
poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(poi_prompt_df, on="location_id", how="left")

# ---- diagnostics ----
print(get_poi_context_for_prompt(sp_sampled2, poi)["addr_poi_dist_km"].describe())
print("nearby_places in columns:", "nearby_places" in sp_sampled2.columns)
print(sp_sampled2["nearby_places"].notna().value_counts(dropna=False))
print(sp_sampled2[["location_id", "nearby_places"]].head(5))

# ---- save ----
SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)
print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

In [21]:
# Step 3: Display results
print("\n" + "="*70)
print("Final sp_sampled2 with POI features:")
print("="*70)
print("Columns:", sp_sampled2.columns.tolist())
print("\nSample rows:")
print(sp_sampled2.head())

# Show POI statistics
poi_cols = [c for c in sp_sampled2.columns if '_cnt' in c or 'dist_' in c]
print("\n" + "="*70)
print("POI Feature Statistics:")
print("="*70)
#print(sp_sampled2[poi_cols].describe())# Step 3: Display results
print("\n" + "="*70)
print("Final sp_sampled2 with POI features:")
print("="*70)
print("Columns:", sp_sampled2.columns.tolist())
print("\nSample rows:")
print(sp_sampled2.head())

# Show POI statistics
poi_cols = [c for c in sp_sampled2.columns if '_cnt' in c or 'dist_' in c]
print("\n" + "="*70)
print("POI Feature Statistics:")
print("="*70)


Final sp_sampled2 with POI features:
Columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode', 'nearby_places']

Sample rows:
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.1436

In [22]:
import ast
import pandas as pd

def tokens_compact_1week(
    df_u: pd.DataFrame,
    max_events: int=50,
    prec: int=4
) -> list[str]:
    """
    Build natural-language mobility tokens from up to one week of staypoints,
    inserting a date header at the beginning of each day.
    """
    # --------------------------------------------------
    # 1. detect duration column
    # --------------------------------------------------
    duration_col = None
    for c in df_u.columns:
        if c.lower() in ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]:
            duration_col = c
            break

    # --------------------------------------------------
    # 2. select usable columns
    # --------------------------------------------------
    use_cols = [
        "started_at", "dow", "hour_bin", "location_id",
        "lon", "lat", "mode",
        "city", "neighbourhood", "road", "nearby_places",
        "postcode"
    ]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()

    # --------------------------------------------------
    # 3. datetime handling + ordering + safety cap
    # --------------------------------------------------
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at")

    if len(df) > max_events:
        df = df.head(max_events)

    # --------------------------------------------------
    # 4. build tokens with date headers
    # --------------------------------------------------
    toks = []
    current_date = None

    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()

        # ---- insert date header when date changes ----
        if date_str != current_date:
            toks.append(f"Date: {date_str}")
            current_date = date_str

        # ---- time info ----
        hhmm = t.strftime("%H:%M")
        dow = int(getattr(r, "dow", -1))
        hb  = int(getattr(r, "hour_bin", -1))

        # ---- location ----
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        def clean_addr_part(s):
            if pd.isna(s):
                return None
            s = str(s).strip()
            if s.lower() in ["nan", "none", "-", ""]:
                return None
            return s
        addr_parts = [
            clean_addr_part(getattr(r, "road", None)),
            clean_addr_part(getattr(r, "neighbourhood", None)),
            clean_addr_part(getattr(r, "city", None)),
        ]
        addr_parts = [p for p in addr_parts if p is not None]
        addr = ", ".join(addr_parts) if addr_parts else "unknown address"
        # add postcode
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode is not None:
            addr = f"{addr}, postcode {postcode}"
        #addr = addr.append("postcode " + str(getattr(r, "postcode", "unknown"))) if hasattr(r, "postcode") else addr


        # # ---- nearby POIs ----
        # nearby_raw = getattr(r, "nearby_places", None)
        # if isinstance(nearby_raw, str):
        #     try:
        #         nearby = ast.literal_eval(nearby_raw)
        #     except (ValueError, SyntaxError):
        #         nearby = []
        # elif isinstance(nearby_raw, list):
        #     nearby = nearby_raw
        # else:
        #     nearby = []

        # nearby_str = "; ".join(nearby[:5]) if nearby else "no nearby places listed"

        # ---- nearby POIs ----
        nearby_raw = getattr(r, "nearby_places", None)

        if isinstance(nearby_raw, list):
        # legacy case: list of strings
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str):
         # current correct case: already a string
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"


        # ---- mode & duration ----
        mode = str(getattr(r, "mode", "unknown")).lower()
        dur_val = getattr(r, duration_col, 0) if duration_col else 0
        dur = int(round(float(dur_val))) if pd.notna(dur_val) else 0

        # ---- final sentence ----
        toks.append(
            f"At {hhmm}, on weekday {dow} (hour block {hb}), "
            f"the user was at coordinates ({lat}, {lon}). "
            f"The address is {addr}. "
            f"The transport mode was {mode}, with an activity duration of {dur} minutes. "
            f"Nearby places include: {nearby_str}."
        )

    return toks

In [23]:
# usage with poi details
toks = tokens_compact_1week(
    df_u=sp_sampled2,
    max_events=40,
    prec=4
)

print(toks[:8])

['Date: 2019-09-02', 'At 20:21, on weekday 0 (hour block 20), the user was at coordinates (46.936, 7.4314). The address is Chutzenstrasse, Bern, postcode 3007. The transport mode was walk, with an activity duration of 503 minutes. Nearby places include: 0.054 km to the north-west: Residential Bahnhof Weissenbühl; 0.06 km to the north-east: Shopping Nido; 0.062 km to the west: Shopping; 0.062 km to the west: Shopping; 0.064 km to the west: Transportation Weissenbühl Bahnhof.', 'Date: 2019-09-03', 'At 05:24, on weekday 1 (hour block 5), the user was at coordinates (47.2148, 7.5201). The address is Vogelherdstrasse, Roamer, postcode 4513. The transport mode was car, with an activity duration of 615 minutes. Nearby places include: 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.088 km to the

### Update prompts Version 1 24 Mar 2026

In [ ]:
# ---------------------------------------------------------------------------
# Build prompts for age and household size prediction.  update on 24 Mar 2026 
# ----------------------------------

from textwrap import dedent

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
sp_prompt = sp_sampled2.copy()
sp_prompt["date_str"] = (
    pd.to_datetime(sp_prompt["started_at"], errors="coerce")
      .dt.tz_localize(None)
      .dt.date
      .astype(str)
)
print(sp_prompt[["started_at", "date", "date_str"]].head(5))

# build valid user-day table from existing rows only
user_day = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
)
user_day = (
    user_day
    .groupby("user_id", group_keys=False)
    #.sample(n=2, random_state=RANDOM_SEED)
    #.rename(columns={"date_str": "sample_date"})
)

#print("user_day size:", len(user_day))   # MUST be > 0

rows_prompts = []

user_dates = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
    .groupby("user_id")["date_str"]
    .apply(list)
)

print(f"user_dates sample: {len(user_dates)}")

for user_id, dates in user_dates.items():
    
    df_u = sp_prompt[

        (sp_prompt["user_id"] == user_id) &
        (sp_prompt["date_str"].isin(dates))
    ]
    print(f"Processing user {user_id} for dates {dates}, df_u shape: {df_u.shape}")


    toks = tokens_compact_1week(df_u, max_events=MAX_EVENTS, prec=PREC)
    if not toks:
        continue
    prompt = (
        f"User: {user_id}\n\n"
        f"Mobility record for one week in Switzerland:\n"
        "Task:\n"
        "Based only on the mobility behaviour and nearby spatial context described below,"
        "infer the user's most likely household size and age group based on swiss census data and situation\n\n"
        "household size: 1, 2,3, 4, 5+, age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say\n\n"
        " Guidelines:\n"
        "- Use only the information explicitly provided in the mobility record.\n"
        "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
        "- Do not assume any unobserved personal attributes.\n"
        "- keep a neutral and factual tone.\n\n"
        "Output format:\n"
        "1. Reasoning (no more than 150 words).\n"
        "2. One final line in strict JSON format, for example:\n"
        " {\"household_size\": \"4\", \"age_group\": \"45-54\", \"gender\": \"other\"}\n\n"
        "Mobility evidance:\n"
        + "\n".join(toks)
    )
    
    # store results as a dictionary
    rows_prompts.append({
        "user_id": user_id,
        "date": dates,
        "prompt": prompt})

# Prompts should be keep the same for every run, every model should get the same prompts for fair comparison, 
# so the name should reflect that they are fixed and deterministic prompts based on the data, not something that 
# changes with model or run, update on 24 Mar 2026.
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_update_v4.json")

PROMPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(PROMPTS_OUT, "w", encoding="utf-8") as f:
    for r in rows_prompts:
        f.write(r["prompt"])
        f.write("\n\n" + "="*80 + "\n\n")

print(f"Saved {len(rows_prompts)} prompts -> {PROMPTS_OUT}")

# print sample
for r in rows_prompts[:1]:
    print("\n--- Sample Prompt ---")
    print(r["prompt"][:1000] + "\n...")   

                        started_at       date    date_str
0 2019-10-22 09:39:28.773999+00:00 2019-10-22  2019-10-22
1 2019-10-23 06:35:32.236999+00:00 2019-10-23  2019-10-23
2 2019-10-23 09:02:58.773999+00:00 2019-10-23  2019-10-23
3 2019-10-23 09:53:18.151999+00:00 2019-10-23  2019-10-23
4 2019-10-23 16:44:43.608999+00:00 2019-10-23  2019-10-23
user_dates sample: 2102
Processing user AAGAF for dates ['2019-10-22', '2019-10-23', '2019-10-24', '2019-10-25', '2019-10-26', '2019-10-27', '2019-10-28'], df_u shape: (23, 24)
Processing user AAINS for dates ['2019-11-13', '2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2019-11-18', '2019-11-19'], df_u shape: (34, 24)
Processing user AAQME for dates ['2019-12-10', '2019-12-11', '2019-12-12', '2019-12-13', '2019-12-14', '2019-12-15', '2019-12-16'], df_u shape: (23, 24)
Processing user AARWP for dates ['2019-09-18', '2019-09-19', '2019-09-20', '2019-09-21', '2019-09-22', '2019-09-23', '2019-09-24'], df_u shape: (35, 24)
Processing user 

Clean the prompts 

In [ ]:
import json
import re
from pathlib import Path
IN_PATH  = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_update_v4.json")
OUT_PATHh = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")
# prompts_v3_1week_age_householdsize_04Mar2026_qwen_update_v4

def clean_prompt_text(text: str) -> str:
    """
    Remove markdown headers, separators, examples, and keep
    only instruction + mobility evidence.
    """
    # 1. remove separator lines ---
    text = re.sub(r"^\s*---\s*$", "", text, flags=re.MULTILINE)

    # 2. remove markdown headers like ## Task Overview
    text = re.sub(r"^\s*## .*?$", "", text, flags=re.MULTILINE)

    # 3. remove Example block
    text = re.sub(
        r'## Example:.*?---',
        '',
        text,
        flags=re.DOTALL
    )

    # 4. collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

cleaned_rows = []

with open(IN_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

blocks = [b.strip() for b in re.split(r"\n{2,}={80,}\n{2,}", raw) if b.strip()]

# apply cleaning
cleaned_blocks = [clean_prompt_text(b) for b in blocks if clean_prompt_text(b)]

# write cleaned jsonl
with open(OUT_PATHh, "w", encoding="utf-8") as f:
    for b in cleaned_blocks:
        f.write(b)
        f.write("\n\n"+ "="*80 + "\n\n")

print(f"Cleaned prompts saved: {len(cleaned_blocks)} -> {OUT_PATHh}")

Cleaned prompts saved: 2102 -> /data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt


## 3. Predict Sociodemographics using "Deepseek 7B" model based on the prompts we just generated 

#### Test GPU, decide Model version/ Model name without finetuing verson

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
 
model_name = "deepseek-ai/deepseek-llm-7b-chat"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Proper 4-bit config
bnb_config = BitsAndBytesConfig( # the parameter are actually in bitandbytescofig, not in the model loading.
    # all the parameters are for 4bit .
    # we can try later when doing the fine tuning.
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained( # the parameters are not belong to here, that's why we need to remove them, 
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ DeepSeek ready.")

Loading weights: 100%|██████████| 339/339 [00:02<00:00, 157.49it/s, Materializing param=model.norm.weight]                              


✅ Qwen ready.


In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

# --------------------------------------
# Config
# --------------------------------------

model_name = "deepseek-ai/deepseek-llm-7b-chat"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_deepseek_age,householdsize_24Mar2026_v1withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_deepseek_age,householdsize_24Mar2026_v1withrationale.csv")

SEP = "=" * 80

MAX_PROMPT_CHARS = 30000
# MAX_PROMPT_CHARS = 30000  # DeepSeek 7B can handle up to xxk tokens, avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages =[
                {"role": "system",
                 "content": (
                     "You are a mobility behavior and socioeconomic status inference analyst."
                     "return only valid JSON with keys: age_group, gender, household_size, household_income_level."
                     "Based only on the mobility behaviour and nearby spatial context described below,"
                     # "infer the user's most likely household income level based on swiss census data and situation\n\n"
                     "Please choose exactly one of the following categories household size,age_group, gender:\n"
                     "household size: 1, 2,3, 4, 5+, age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say\n\n"
                     " Guidelines:\n"
                     "- Use only the information explicitly provided in the mobility record.\n"
                     "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
                     "- Do not assume any unobserved personal attributes.\n"
                     "- keep a neutral and factual tone.\n\n"
                     "Output format:\n"
                     "1. Reasoning (no more than 100 words).\n" # need to adjust the parameters for rationale length
                     "2. One final line in strict JSON format, for example:\n"
                     " {\"household_size\": \"4\", \"age_group\": \"45-54\", \"gender\": \"other\", \"predictionrationale\":\"explain how the prediction was made.\"}\n\n"
                     "- prediction_rationale must be a list of 3-6 short bullet-like strings, each describing a concrete mobility/semantic evidence. \n "
                     "- Do not provide step by step reasoning, or hidden chain of thought"
                     "- Just provide evidence-style justification.\n"
                     "- prediction_confidence must be one of :low, medium, high.\n"
                     " no extra keys, no markdown, no commentary outside josn"
                     " use chain of thought for inference, do not just guess based on one single evidence, but try to find multiple evidences and weigh them together for a more robust inference."

                 )
                 },
            ]

            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True)

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # give space for rationale
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05 # to encourage more diverse outputs
            )

            signal.alarm(0)
            
            # decode only the generated part, avoid mixing prompt + answer
            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]  

            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                #json_str = json_match.group(0)
                #try:
                    #result = json.loads(json_str)


                # need to fix 
                result = json.loads(json_match.group(0))
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

📦 Loaded 2102 prompts
🔁 Found 2060 completed users
⏭️ Skipping AAGAF
⏭️ Skipping AAINS
⏭️ Skipping AAQME
⏭️ Skipping AARWP
⏭️ Skipping ABARC
⏭️ Skipping ABECR
⏭️ Skipping ABENL
⏭️ Skipping ABISR
⏭️ Skipping ABLPH
⏭️ Skipping ACSUF
⏭️ Skipping ACWVM
⏭️ Skipping ADCMH
⏭️ Skipping AEBWY
⏭️ Skipping AEGHJ
⏭️ Skipping AEWVV
⏭️ Skipping AFNCT
⏭️ Skipping AFUVG
⏭️ Skipping AGHUY
⏭️ Skipping AHTYK
⏭️ Skipping AJJJR
⏭️ Skipping AJJNX
⏭️ Skipping AJKBY
⏭️ Skipping AJQGV
⏭️ Skipping AKGPG
⏭️ Skipping AKMIL
⏭️ Skipping AKOZE
⏭️ Skipping AKQKS
⏭️ Skipping ALKPX
⏭️ Skipping ALSMT
⏭️ Skipping AMGUM
⏭️ Skipping AMGZL
⏭️ Skipping AMYMD
⏭️ Skipping ANEFF
⏭️ Skipping ANMKC
⏭️ Skipping ANQHI
⏭️ Skipping AOVGU
⏭️ Skipping AOVLF
⏭️ Skipping APKPN
⏭️ Skipping APWYR
⏭️ Skipping AQFEA
⏭️ Skipping AQQGE
⏭️ Skipping AQREK
⏭️ Skipping ARFNG
⏭️ Skipping ARMZF
⏭️ Skipping ARZDI
⏭️ Skipping ASUKL
⏭️ Skipping ASXOD
⏭️ Skipping ATBSI
⏭️ Skipping ATCBP
⏭️ Skipping ATEAN
⏭️ Skipping ATKDY
⏭️ Skipping AUXLS
⏭️ Skipping A

prediction with rationale  need to check

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------
# Config
# --------------------------------------

model_name = "deepseek-ai/deepseek-llm-7b-chat"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_deepseek_age,householdsize_24Mar2026_v2withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_deepseek_age,householdsize_24Mar2026_v2withrationale.csv")

SEP = "=" * 80

MAX_PROMPT_CHARS = 30000

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages =[
                {"role": "system",
                 "content": (
                     "You are a mobility behavior and socioeconomic status inference analyst."
                     "return only valid JSON with keys: age_group, gender, household_size, household_income_level."
                     "Based only on the mobility behaviour and nearby spatial context described below,"
                     # "infer the user's most likely household income level based on swiss census data and situation\n\n"
                     "Please choose exactly one of the following categories (monthly household income in CHF),age_group, gender, household_size:\n"
                     "<4000, 4001-8000, 8001-12000, 12001-16000, 16001+. age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say; household_size: 1, 2, 3, 4, 5+\n\n"
                     " Guidelines:\n"
                     " this project conducted in switzerland, please consider the swiss context when making inference. use the swiss geographic context, like the distribution of urban and rural areas, like event happened in zentrum zurich, oerlikon zurich,  public transportation networks, and local economic activities. \n"
                     "- Use only the information explicitly provided in the mobility record.\n"
                     "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
                     "- keep a neutral and factual tone.\n\n"
                     "Output format:\n"
                     "1. Reasoning (no more than 150 words).\n" # need to adjust the parameters for rationale length
                     "2. One final line in strict JSON format, for example:\n"
                     " {\"household_income_level\": \"4001-8000\", \"age_group\": \"45-54\", \"gender\": \"other\", \"household_size\": 3, \"prediction_rationale\": [\"Evidence 1\", \"Evidence 2, give us the real rationale\"], \"prediction_confidence\": \"high\"}\n\n"
                     "- prediction_rationale must be a list of 3-6 short bullet-like strings, each describing a concrete mobility/semantic evidence. \n "
                     "- Provide step by step reasoning, or chain of thought"
                     "- Just provide evidence-style justification.\n"
                     "- prediction_confidence must be one of :low, medium, high.\n"
                     "no extra keys, no markdown, no commentary outside josn"

                 )
                 },
            ]

            tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True)

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # give space for rationale
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05  # to encourage more diverse outputs
            )

            signal.alarm(0)
            
            # decode only the generated part, avoid mixing prompt + answer
            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]  

            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                #json_str = json_match.group(0)
                #try:
                    #result = json.loads(json_str)


                # need to fix 
                result = json.loads(json_match.group(0))
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

📦 Loaded 2102 prompts
🔁 Found 1755 completed users
⏭️ Skipping AAGAF
⏭️ Skipping AAINS
⏭️ Skipping AAQME
⏭️ Skipping AARWP
⏭️ Skipping ABARC
⏭️ Skipping ABECR
⏭️ Skipping ABENL
⏭️ Skipping ABISR
⏭️ Skipping ABLPH
⏭️ Skipping ACSUF
⏭️ Skipping ACWVM
⏭️ Skipping ADCMH
⏭️ Skipping AEBWY
⏭️ Skipping AEGHJ
⏭️ Skipping AEWVV
⏭️ Skipping AFNCT
⏭️ Skipping AFUVG

🔮 [18/2102] Predicting AGHUY
❌ Error: Expecting value: line 12 column 1 (char 873)
⏭️ Skipping AHTYK
⏭️ Skipping AJJJR
⏭️ Skipping AJJNX
⏭️ Skipping AJKBY
⏭️ Skipping AJQGV
⏭️ Skipping AKGPG

🔮 [25/2102] Predicting AKMIL
✅ Done
⏭️ Skipping AKOZE

🔮 [27/2102] Predicting AKQKS
✅ Done
⏭️ Skipping ALKPX
⏭️ Skipping ALSMT
⏭️ Skipping AMGUM
⏭️ Skipping AMGZL
⏭️ Skipping AMYMD
⏭️ Skipping ANEFF
⏭️ Skipping ANMKC
⏭️ Skipping ANQHI

🔮 [36/2102] Predicting AOVGU
✅ Done
⏭️ Skipping AOVLF
⏭️ Skipping APKPN
⏭️ Skipping APWYR
⏭️ Skipping AQFEA

🔮 [41/2102] Predicting AQQGE
✅ Done
⏭️ Skipping AQREK
⏭️ Skipping ARFNG
⏭️ Skipping ARMZF
⏭️ Skipping ARZ

## Evaluation

#### this precedure is just for income level

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl")
#preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl

df = pd.read_json(raw_path, lines=True)
print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income, age group, household size, gender extraction functions
# --------------------------------------
def extract_from_text(text):
    if pd.isna(text):
        return None

    try:
        data = json.loads(text)
        income_level = data.get("income_level", "")
        if income_level:
            if income_level in [">=16000", ">=16001", "16001+", "16000+", "more than 16000", "above 16000", "at least 16000", "16000+"]:
                return ">16000"
            return income_level
    except Exception:
        pass

    text = str(text).lower()

    norm = (
        text.lower()
            .replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    if re.search(
        r'(>=\s*16000|>=\s*16001|16001\+|16000\+|'
        r'more\s+than\s+16000|above\s+16000|at\s+least\s+16000)',
        norm
    ):
        return ">16000"

    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}[,.]?\d{3})\s*[-–]\s*(\d{1,2}[,.]?\d{3})', norm)

    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    else:
        return None

# --------------------------------------
# Apply extraction
# --------------------------------------
TEXT_COL = "household_income_level"
df["household_income_level"] = df[TEXT_COL].apply(extract_from_text)

print(df.columns.tolist())
print(df.head(3))
print(df["household_income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned CSV
# -----------------------------------------
df_clean = (
    df[["user_id", "household_income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.jsonl")
df_clean.to_json(out_path, orient="records", lines=True)
out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")
# preds_qwen_income_22Feb2026_v2withrationale.csv
df_clean.to_csv(out_path, index=False)

print(f"✅ Clean predictions saved to: {out_path}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(10))

Loaded raw predictions: 2062 rows
['household_income_level', 'age_group', 'gender', 'income_rationale', 'income_confidence', 'user_id']
  household_income_level age_group  gender  \
0            12001-16000     35-44  female   
1            12001-16000     35-44  female   
2             8001-12000     25-34  female   

                                    income_rationale income_confidence user_id  
0  ['frequent visits to high-cost business areas'...              high   AAGAF  
1  ['visited multiple high-cost leisure and enter...              high   AAINS  
2  ['frequent visits to middle to high-income nei...            medium   AAQME  
household_income_level
8001-12000     1206
12001-16000     856
Name: count, dtype: int64
✅ Clean predictions saved to: /data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv
📊 Rows: 2062
  user_id household_income_level
0   AAGAF            12001-16000
1   AAINS            12001-16000
2   AAQME             8001-12000
3   AARW

In [4]:
import pandas as pd
import json
import re
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================
# Define the column containing raw model output
# If the data is in a single column (like "prediction" or "raw_output"):
TEXT_COL = "prediction"  # ← Change this to your actual column name

# If each field is already in separate columns, set TEXT_COL = None
# and define the column names:
SEPARATE_COLS = False  # Set to True if data is already in separate columns
AGE_COL = "age_group"
GENDER_COL = "gender"
INCOME_COL = "household_income_level"
HOUSEHOLD_COL = "household_size"

# ============================================================================
# LOAD RAW PREDICTIONS
# ============================================================================
raw_path = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl")
df = pd.read_json(raw_path, lines=True)
print(f"Loaded raw predictions: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst row sample:")
print(df.iloc[0])

# ============================================================================
# INCOME BIN MAPPING
# ============================================================================
def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    return None

# ============================================================================
# GENERIC JSON SAFE LOADER
# ============================================================================
def safe_json_load(text):
    if pd.isna(text):
        return {}
    try:
        return json.loads(text)
    except Exception:
        return {}

# ============================================================================
# GENDER CLEANING
# ============================================================================
def clean_gender(text):
    if pd.isna(text):
        return None
    text = str(text).lower()
    if "female" in text or text == "f":
        return "female"
    if "male" in text or text == "m":
        return "male"
    if "other" in text:
        return "other"
    return None

# ============================================================================
# AGE GROUP CLEANING
# ============================================================================
VALID_AGE_GROUPS = [
    "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
]

def clean_age(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    for group in VALID_AGE_GROUPS:
        if group in text:
            return group

    # Catch patterns like "between 26 and 35"
    m = re.search(r'(\d{2})\s*[-–]\s*(\d{2})', text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    return None

# ============================================================================
# HOUSEHOLD SIZE CLEANING
# ============================================================================
def clean_household_size(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    if "1" in text:
        return "1"
    if "2" in text:
        return "2"
    if "3" in text:
        return "3"
    if "4" in text:
        return "4"
    if "5" in text or "5+" in text or "more than 5" in text:
        return "5+"

    return None

# ============================================================================
# INCOME CLEANING
# ============================================================================
def clean_income(text):
    if pd.isna(text):
        return None

    # Try JSON first
    try:
        data = json.loads(text)
        income = data.get("income_level", "")
        if income:
            income = str(income)
            if "16000" in income or "16001" in income:
                return ">16000"
            return income
    except Exception:
        pass

    text = str(text).lower()
    norm = (
        text.replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    # High income  
    if re.search(r'(>=\s*16000|16000\+|more\s+than\s+16000|above\s+16000)', norm):
        return ">16000"

    # Between pattern
    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}\d{3})\s*[-–]\s*(\d{1,2}\d{3})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    # Direct bin mention
    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

# ============================================================================
# EXTRACT ALL FIELDS
# ============================================================================
def extract_all(text):
    """
    Extract all fields from a single text/JSON column.
    Returns 4 values: [age_group, gender, income, household_size]
    """
    data = safe_json_load(text)

    # Try JSON fields first
    age = data.get("age_group", None)
    gender = data.get("gender", None)
    income = data.get("income_level", None) or data.get("household_income_level", None)
    household_size = data.get("household_size", None)

    # Fallback to raw text parsing
    if age is None:
        age = clean_age(text)
    else:
        age = clean_age(age)

    if gender is None:
        gender = clean_gender(text)
    else:
        gender = clean_gender(gender)

    if income is None:
        income = clean_income(text)
    else:
        income = clean_income(income)

    if household_size is None:
        household_size = clean_household_size(text)
    else:
        household_size = clean_household_size(household_size)

    return pd.Series([age, gender, income, household_size])

# ============================================================================
# APPLY EXTRACTION
# ============================================================================
if not SEPARATE_COLS:
    # Case 1: Data is in a single column
    if TEXT_COL in df.columns:
        print(f"\n✓ Found column: {TEXT_COL}")
        print(f"Sample data:\n{df[TEXT_COL].head(2)}")
        
        # Apply extraction
        df[["age_group", "gender", "household_income_level", "household_size"]] = \
            df[TEXT_COL].apply(extract_all)
    else:
        print(f"\n Error: Column '{TEXT_COL}' not found!")
        print(f"Available columns: {df.columns.tolist()}")
        print("\nPlease update TEXT_COL to the correct column name")
        print("Example: TEXT_COL = 'prediction'")
        exit(1)
else:
    # Case 2: Data is already in separate columns
    print(f"\n✓ Using separate columns mode")
    if AGE_COL in df.columns:
        df["age_group"] = df[AGE_COL].apply(clean_age)
    if GENDER_COL in df.columns:
        df["gender"] = df[GENDER_COL].apply(clean_gender)
    if INCOME_COL in df.columns:
        df["household_income_level"] = df[INCOME_COL].apply(clean_income)
    if HOUSEHOLD_COL in df.columns:
        df["household_size"] = df[HOUSEHOLD_COL].apply(clean_household_size)

# ============================================================================
# PRINT STATISTICS
# ============================================================================
print("\n" + "="*70)
print("VALUE COUNTS")
print("="*70)

print("\nAge Group:")
print(df["age_group"].value_counts(dropna=False))

print("\nGender:")
print(df["gender"].value_counts(dropna=False))

print("\nIncome Level:")
print(df["household_income_level"].value_counts(dropna=False))

print("\nHousehold Size:")
print(df["household_size"].value_counts(dropna=False))

# Check for missing values
print("\n" + "="*70)
print("MISSING VALUES")
print("="*70)
missing = {
    "age_group": df["age_group"].isna().sum(),
    "gender": df["gender"].isna().sum(),
    "household_income_level": df["household_income_level"].isna().sum(),
    "household_size": df["household_size"].isna().sum()
}
for field, count in missing.items():
    pct = count / len(df) * 100
    print(f"{field:30s}: {count:5d} ({pct:5.2f}%)")

# ============================================================================
# SAVE CLEANED DATA
# ============================================================================
df_clean = (
    df[["user_id", "age_group", "gender", "household_income_level", "household_size"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_csv = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")
df_clean.to_csv(out_csv, index=False)

out_jsonl = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.jsonl")
df_clean.to_json(out_jsonl, orient="records", lines=True)

print("\n" + "="*70)
print("CLEANING COMPLETED")
print("="*70)
print(f"Output CSV:   {out_csv}")
print(f"Output JSONL: {out_jsonl}")
print(f"Total rows:   {len(df_clean)}")
print("\nFirst 10 rows:")
print(df_clean.head(10))

Loaded raw predictions: 2038 rows
Columns: ['household_income_level', 'age_group', 'gender', 'household_size', 'prediction_rationale', 'prediction_confidence', 'user_id', 'reasoning']

First row sample:
household_income_level                                            4001-8000
age_group                                                             35-44
gender                                                               female
household_size                                                          4.0
prediction_rationale      [High frequency visits to supermarkets and gro...
prediction_confidence                                                medium
user_id                                                               AAGAF
reasoning                                                               NaN
Name: 0, dtype: object

 Error: Column 'prediction' not found!
Available columns: ['household_income_level', 'age_group', 'gender', 'household_size', 'prediction_rationale', 'prediction_con

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# FILE PATHS
# ============================================================================
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")

# ============================================================================
# Load datasets
# ============================================================================
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

print("\n" + "="*70)
print("GROUND TRUTH COLUMNS")
print("="*70)
print(gt.columns.tolist())
print("\nFirst row:")
print(gt.head(1))

print("\n" + "="*70)
print("PREDICTION COLUMNS")
print("="*70)
print(pred.columns.tolist())
print("\nFirst row:")
print(pred.head(1))

# ============================================================================
# SELECT & RENAME COLUMNS
# ============================================================================
# Ground truth columns
gt = gt[["user_id", "age_group", "household_size_group", "income_level", "gender"]].copy()

# Prediction columns
pred = pred[["user_id", "age_group", "gender", "household_income_level", "household_size"]].copy()

# ============================================================================
# NORMALIZE HOUSEHOLD SIZE
# ============================================================================
def normalize_household_size(x):
    """Convert household size to standardized format: '1', '2', '3', '4', '5+'"""
    if pd.isna(x):
        return None
    
    x_str = str(x).strip().lower()
    
    # Handle '5+' or 'more than 5'
    if '5+' in x_str or 'more than 5' in x_str or x_str == '5+':
        return "5+"
    
    # Try to convert to number
    try:
        x_num = float(x_str)
        if x_num >= 5:
            return "5+"
        elif x_num >= 1 and x_num <= 4:
            return str(int(x_num))
        else:
            return None
    except ValueError:
        # Try to extract number
        import re
        match = re.search(r'\d+', x_str)
        if match:
            num = int(match.group())
            if num >= 5:
                return "5+"
            elif num >= 1 and num <= 4:
                return str(num)
        return None

# Apply normalization to both datasets
gt["household_size_norm"] = gt["household_size_group"].apply(normalize_household_size)
pred["household_size_norm"] = pred["household_size"].apply(normalize_household_size)

print("\n" + "="*70)
print("NORMALIZED HOUSEHOLD SIZE")
print("="*70)
print("Ground Truth:")
print(gt["household_size_norm"].value_counts(dropna=False))
print("\nPredictions:")
print(pred["household_size_norm"].value_counts(dropna=False))

# ============================================================================
# MERGE GT AND PREDICTIONS
# ============================================================================
merged = pd.merge(
    gt[["user_id", "age_group", "gender", "household_size_norm", "income_level"]],
    pred[["user_id", "age_group", "gender", "household_size_norm", "household_income_level"]],
    on="user_id",
    suffixes=("_true", "_pred"),
    how="inner"
)

print(f"\n✓ Merged {len(merged)} users")
print("\nMerged columns:")
print(merged.columns.tolist())
print("\nFirst 5 rows:")
print(merged.head(5))

# ============================================================================
# MASKS FOR VALID EVALUATION SAMPLES
# ============================================================================
# Age mask
age_mask = (
    merged["age_group_true"].notna() &
    merged["age_group_pred"].notna()
)

# Household size mask
household_mask = (
    merged["household_size_norm_true"].notna() &
    merged["household_size_norm_pred"].notna()
)

# Gender mask
gender_mask = (
    merged["gender_true"].notna() &
    merged["gender_pred"].notna()
)

# Income mask (exclude "prefer not to say")
income_mask = (
    merged["income_level"].notna() &
    merged["household_income_level"].notna() &
    (merged["income_level"].str.lower() != "prefer not to say")
)

print("\n" + "="*70)
print("EVALUATION SAMPLE SIZES")
print("="*70)
print(f"Age:           {age_mask.sum():5d} / {len(merged)} ({age_mask.sum()/len(merged)*100:.1f}%)")
print(f"Gender:        {gender_mask.sum():5d} / {len(merged)} ({gender_mask.sum()/len(merged)*100:.1f}%)")
print(f"Household:     {household_mask.sum():5d} / {len(merged)} ({household_mask.sum()/len(merged)*100:.1f}%)")
print(f"Income:        {income_mask.sum():5d} / {len(merged)} ({income_mask.sum()/len(merged)*100:.1f}%)")

# ============================================================================
# ACCURACY COMPUTATION
# ============================================================================
age_acc = (
    merged.loc[age_mask, "age_group_true"] == 
    merged.loc[age_mask, "age_group_pred"]
).mean()

gender_acc = (
    merged.loc[gender_mask, "gender_true"] == 
    merged.loc[gender_mask, "gender_pred"]
).mean()

household_acc = (
    merged.loc[household_mask, "household_size_norm_true"] == 
    merged.loc[household_mask, "household_size_norm_pred"]
).mean()

income_acc = (
    merged.loc[income_mask, "income_level"] == 
    merged.loc[income_mask, "household_income_level"]
).mean()

# ============================================================================
# MAJORITY-CLASS BASELINES
# ============================================================================
def majority_baseline(y_true):
    """Calculate accuracy if always predicting the majority class"""
    if len(y_true) == 0:
        return 0.0
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(merged.loc[age_mask, "age_group_true"])
gender_baseline = majority_baseline(merged.loc[gender_mask, "gender_true"])
household_baseline = majority_baseline(merged.loc[household_mask, "household_size_norm_true"])
income_baseline = majority_baseline(merged.loc[income_mask, "income_level"])

# ============================================================================
# OUTPUT RESULTS
# ============================================================================
print("\n" + "="*70)
print("📊 ACCURACY RESULTS")
print("="*70)
print(f"Age group:        {age_acc:.3f}  (baseline: {age_baseline:.3f})  [n={age_mask.sum()}]")
print(f"Gender:           {gender_acc:.3f}  (baseline: {gender_baseline:.3f})  [n={gender_mask.sum()}]")
print(f"Household size:   {household_acc:.3f}  (baseline: {household_baseline:.3f})  [n={household_mask.sum()}]")
print(f"Income level:     {income_acc:.3f}  (baseline: {income_baseline:.3f})  [n={income_mask.sum()}]")

# ============================================================================
# SAVE MERGED COMPARISON TABLES
# ============================================================================
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path("/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv")
merged.loc[income_mask].to_csv(filtered_income_path, index=False, encoding="utf-8")

print(f"\n✓ Detailed comparison saved to: {output_path}")
print(f"✓ Income-eval subset saved to: {filtered_income_path}")

# ============================================================================
# CONFUSION MATRICES
# ============================================================================
print("\n" + "="*70)
print("CONFUSION MATRICES")
print("="*70)

# Age confusion matrix
if age_mask.sum() > 0:
    age_cm_counts = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    age_cm_norm = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    age_cm_path = Path("/data/baliu/python_code/data/age_confusion_matrix_v4_final.csv")
    age_cm_norm.to_csv(age_cm_path)
    
    
    print("\nAge confusion matrix (row-normalized):")
    print(age_cm_norm.round(3))
    print(f"✓ Saved to: {age_cm_path}")

# Gender confusion matrix
if gender_mask.sum() > 0:
    gender_cm_counts = pd.crosstab(
        merged.loc[gender_mask, "gender_true"],
        merged.loc[gender_mask, "gender_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    gender_cm_norm = pd.crosstab(
        merged.loc[gender_mask, "gender_true"],
        merged.loc[gender_mask, "gender_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    gender_cm_path = Path("/data/baliu/python_code/data/gender_confusion_matrix_v4_final.csv")
    gender_cm_norm.to_csv(gender_cm_path)
    
    print("\nGender confusion matrix (row-normalized):")
    print(gender_cm_norm.round(3))
    print(f"✓ Saved to: {gender_cm_path}")

# Household confusion matrix
if household_mask.sum() > 0:
    household_cm_counts = pd.crosstab(
        merged.loc[household_mask, "household_size_norm_true"],
        merged.loc[household_mask, "household_size_norm_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    household_cm_norm = pd.crosstab(
        merged.loc[household_mask, "household_size_norm_true"],
        merged.loc[household_mask, "household_size_norm_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    household_cm_path = Path("/data/baliu/python_code/data/household_confusion_matrix_v4_final.csv")
    household_cm_norm.to_csv(household_cm_path)
    
    print("\nHousehold size confusion matrix (row-normalized):")
    print(household_cm_norm.round(3))
    print(f"✓ Saved to: {household_cm_path}")

# Income confusion matrix
if income_mask.sum() > 0:
    income_cm_counts = pd.crosstab(
        merged.loc[income_mask, "income_level"],
        merged.loc[income_mask, "household_income_level"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    income_cm_norm = pd.crosstab(
        merged.loc[income_mask, "income_level"],
        merged.loc[income_mask, "household_income_level"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    income_cm_path = Path("/data/baliu/python_code/data/income_confusion_matrix_v4_final.csv")
    income_cm_norm.to_csv(income_cm_path)
    
    print("\nIncome confusion matrix (counts):")
    print(income_cm_counts)
    
    print("\nIncome confusion matrix (row-normalized):")
    print(income_cm_norm.round(3))
    print(f"✓ Saved to: {income_cm_path}")

# ============================================================================
# summary
# ============================================================================
print("\n" + "="*70)
print("✅ EVALUATION COMPLETED")
print("="*70)
print(f"Total users evaluated: {len(merged)}")
print(f"\nAccuracy Summary:")
print(f"  Age:       {age_acc:.1%}")
print(f"  Gender:    {gender_acc:.1%}")
print(f"  Household: {household_acc:.1%}")
print(f"  Income:    {income_acc:.1%}")


/tmp/ipykernel_369649/3375008110.py:14: DtypeWarning: Columns (0: postcode_home, 1: education, 2: main_employment, 3: income, 4: household_size, 5: citizen_1, 6: citizen_2, 7: intro_survey_started_at, 8: intro_survey_finished_at, 9: intro_survey_finished, 10: work_status_employed, 11: work_status_self_employed, 12: work_status_unemployed, 13: work_status_apprentice, 14: work_status_student, 15: work_status_retired, 16: work_status_other, 17: own_vehicles_car, 18: own_vehicles_motorbike, 19: own_vehicles_bicycle, 20: car_fuel, 21: car_year, 22: car_size, 23: bike_type_regular, 24: bike_type_ebike_25, 25: bike_type_ebike_45, 26: pt_pass_ga, 27: pt_pass_half_fare, 28: pt_pass_regional_pass, 29: pt_pass_track_7, 30: pt_pass_other, 31: pt_pass_no_pass, 32: freq_cardriver_own_car, 33: freq_cardriver_shared_car, 34: freq_carpass_car_in_hh, 35: freq_carpass_car_pooling, 36: freq_carpass_taxi, 37: freq_carpass_app_based, 38: freq_pt_train, 39: freq_pt_local_pt, 40: freq_bike_own_bike, 41: freq_

Loaded ground truth (90909 rows)
Loaded predictions  (2038 rows)

GROUND TRUTH COLUMNS
['participant_ID', 'age', 'gender', 'wave', 'postcode_home', 'language', 'education', 'main_employment', 'income', 'household_size', 'citizen_1', 'citizen_2', 'intro_survey_started_at', 'intro_survey_finished_at', 'duration_s', 'intro_survey_finished', 'intro_survey_progress', 'work_status_employed', 'work_status_self_employed', 'work_status_unemployed', 'work_status_apprentice', 'work_status_student', 'work_status_retired', 'work_status_other', 'workload_jobs_main', 'workload_jobs_secondary', 'postcode_jobs_main', 'postcode_jobs_secondary', 'own_vehicles_car', 'own_vehicles_motorbike', 'own_vehicles_bicycle', 'car_fuel', 'car_year', 'car_size', 'car_engine', 'bike_type_regular', 'bike_type_ebike_25', 'bike_type_ebike_45', 'pt_pass_ga', 'pt_pass_half_fare', 'pt_pass_regional_pass', 'pt_pass_track_7', 'pt_pass_other', 'pt_pass_no_pass', 'freq_cardriver_own_car', 'freq_cardriver_shared_car', 'freq_carp

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ============================================================================
# FILE PATHS
# ============================================================================
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")

# ============================================================================
# LOAD DATA
# ============================================================================
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print("="*70)
print("HOUSEHOLD INCOME PREDICTION EVALUATION")
print("="*70)
print(f"Ground truth: {len(gt)} rows")
print(f"Predictions:  {len(pred)} rows")

# ============================================================================
# NORMALIZATION FUNCTIONS
# ============================================================================

def normalize_gender(text):
    """Normalize gender to lowercase"""
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    if text in ['female', 'f']:
        return 'female'
    if text in ['male', 'm']:
        return 'male'
    if text in ['other', 'non-binary']:
        return 'other'
    return None


def normalize_age_group(text):
    """
    Normalize age groups to standard 10-year bins.
    
    GT format: "18-24", "25-44", "45-66", "67+"
    Pred format: "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
    
    Map to unified format: "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
    """
    if pd.isna(text):
        return None
    
    text = str(text).strip().lower()
    
    # Extract numbers
    import re
    numbers = re.findall(r'\d+', text)
    
    if not numbers:
        return None
    
    # Get age range
    if len(numbers) >= 2:
        low = int(numbers[0])
        high = int(numbers[1])
    elif len(numbers) == 1:
        low = int(numbers[0])
        high = low
    else:
        return None
    
    # Map to standard bins
    if low <= 25:
        return "18-25"
    elif low <= 35:
        return "26-35"
    elif low <= 45:
        return "36-45"
    elif low <= 55:
        return "46-55"
    elif low <= 65:
        return "56-65"
    else:
        return "66+"


def normalize_household_size(x):
    """Convert household size to standardized format: '1', '2', '3', '4', '5+'"""
    if pd.isna(x):
        return None
    
    x_str = str(x).strip().lower()
    
    # Handle '5+' or 'more than 5'
    if '5+' in x_str or 'more than 5' in x_str or '>5' in x_str or '>=5' in x_str:
        return "5+"
    
    # Try to convert to number
    try:
        x_num = float(x_str)
        if x_num >= 5:
            return "5+"
        elif x_num >= 1 and x_num <= 4:
            return str(int(x_num))
        else:
            return None
    except ValueError:
        # Try to extract number
        import re
        match = re.search(r'\d+', x_str)
        if match:
            num = int(match.group())
            if num >= 5:
                return "5+"
            elif num >= 1 and num <= 4:
                return str(num)
        return None


def normalize_income(text):
    """Normalize income levels to standard format"""
    if pd.isna(text):
        return None
    
    text = str(text).strip().lower()
    
    # Direct matches
    if '<4000' in text or '< 4000' in text or 'less than 4000' in text:
        return "<4000"
    if '4001-8000' in text or '4000-8000' in text:
        return "4001-8000"
    if '8001-12000' in text or '8000-12000' in text:
        return "8001-12000"
    if '12001-16000' in text or '12000-16000' in text:
        return "12001-16000"
    if '>16000' in text or '> 16000' in text or 'more than 16000' in text or '16001' in text:
        return ">16000"
    
    return None


# ============================================================================
# APPLY NORMALIZATION
# ============================================================================
print("\n" + "="*70)
print("NORMALIZING DATA")
print("="*70)

# Ground truth
gt_clean = gt[["user_id", "age_group", "household_size_group", "income_level", "gender"]].copy()
gt_clean["age_norm"] = gt_clean["age_group"].apply(normalize_age_group)
gt_clean["gender_norm"] = gt_clean["gender"].apply(normalize_gender)
gt_clean["household_norm"] = gt_clean["household_size_group"].apply(normalize_household_size)
gt_clean["income_norm"] = gt_clean["income_level"].apply(normalize_income)

# Predictions
pred_clean = pred[["user_id", "age_group", "gender", "household_income_level", "household_size"]].copy()
pred_clean["age_norm"] = pred_clean["age_group"].apply(normalize_age_group)
pred_clean["gender_norm"] = pred_clean["gender"].apply(normalize_gender)
pred_clean["household_norm"] = pred_clean["household_size"].apply(normalize_household_size)
pred_clean["income_norm"] = pred_clean["household_income_level"].apply(normalize_income)

print("✓ Normalization complete")

# Show distributions
print("\n📊 Ground Truth Distributions:")
print(f"Age:       {gt_clean['age_norm'].value_counts().to_dict()}")
print(f"Gender:    {gt_clean['gender_norm'].value_counts().to_dict()}")
print(f"Household: {gt_clean['household_norm'].value_counts().to_dict()}")
print(f"Income:    {gt_clean['income_norm'].value_counts().to_dict()}")

print("\n📊 Prediction Distributions:")
print(f"Age:       {pred_clean['age_norm'].value_counts().to_dict()}")
print(f"Gender:    {pred_clean['gender_norm'].value_counts().to_dict()}")
print(f"Household: {pred_clean['household_norm'].value_counts().to_dict()}")
print(f"Income:    {pred_clean['income_norm'].value_counts().to_dict()}")

# ============================================================================
# MERGE
# ============================================================================
print("\n" + "="*70)
print("MERGING DATA")
print("="*70)

merged = pd.merge(
    gt_clean[["user_id", "age_norm", "gender_norm", "household_norm", "income_norm"]],
    pred_clean[["user_id", "age_norm", "gender_norm", "household_norm", "income_norm"]],
    on="user_id",
    suffixes=("_true", "_pred"),
    how="inner"
)

print(f"✓ Merged {len(merged)} users")

# Sample check
print("\n🔍 Sample of merged data (first 5 rows):")
sample_cols = ["user_id", "age_norm_true", "age_norm_pred", "gender_norm_true", "gender_norm_pred"]
print(merged[sample_cols].head())

# ============================================================================
# EVALUATION MASKS
# ============================================================================
print("\n" + "="*70)
print("CREATING EVALUATION MASKS")
print("="*70)

age_mask = (
    merged["age_norm_true"].notna() &
    merged["age_norm_pred"].notna()
)

gender_mask = (
    merged["gender_norm_true"].notna() &
    merged["gender_norm_pred"].notna()
)

household_mask = (
    merged["household_norm_true"].notna() &
    merged["household_norm_pred"].notna()
)

income_mask = (
    merged["income_norm_true"].notna() &
    merged["income_norm_pred"].notna()
)

print(f"Age samples:       {age_mask.sum():5d} / {len(merged)} ({age_mask.sum()/len(merged)*100:.1f}%)")
print(f"Gender samples:    {gender_mask.sum():5d} / {len(merged)} ({gender_mask.sum()/len(merged)*100:.1f}%)")
print(f"Household samples: {household_mask.sum():5d} / {len(merged)} ({household_mask.sum()/len(merged)*100:.1f}%)")
print(f"Income samples:    {income_mask.sum():5d} / {len(merged)} ({income_mask.sum()/len(merged)*100:.1f}%)")

# ============================================================================
# COMPUTE ACCURACY
# ============================================================================
print("\n" + "="*70)
print("COMPUTING ACCURACY")
print("="*70)

def compute_accuracy(y_true, y_pred, mask):
    if mask.sum() == 0:
        return 0.0
    return (y_true[mask] == y_pred[mask]).mean()

age_acc = compute_accuracy(merged["age_norm_true"], merged["age_norm_pred"], age_mask)
gender_acc = compute_accuracy(merged["gender_norm_true"], merged["gender_norm_pred"], gender_mask)
household_acc = compute_accuracy(merged["household_norm_true"], merged["household_norm_pred"], household_mask)
income_acc = compute_accuracy(merged["income_norm_true"], merged["income_norm_pred"], income_mask)

print(f"Age accuracy:       {age_acc:.3f} ({age_acc*100:.1f}%) [n={age_mask.sum()}]")
print(f"Gender accuracy:    {gender_acc:.3f} ({gender_acc*100:.1f}%) [n={gender_mask.sum()}]")
print(f"Household accuracy: {household_acc:.3f} ({household_acc*100:.1f}%) [n={household_mask.sum()}]")
print(f"Income accuracy:    {income_acc:.3f} ({income_acc*100:.1f}%) [n={income_mask.sum()}]")

# ============================================================================
# CONFUSION MATRIX PLOTTING FUNCTION
# ============================================================================

def plot_confusion_matrix(y_true, y_pred, mask, title, save_path):
    """
    Plot confusion matrix heatmap similar to XGBoost visualization.
    """
    if mask.sum() == 0:
        print(f"⚠️  Skipping {title}: No valid samples")
        return None, None
    
    # Create confusion matrices
    cm_counts = pd.crosstab(
        y_true[mask],
        y_pred[mask],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    cm_norm = pd.crosstab(
        y_true[mask],
        y_pred[mask],
        rownames=['True'],
        colnames=['Predicted'],
        normalize='index'
    )
    
    # Custom colormap (purple to yellow like your image)
    colors = ['#2d1b3d', '#3a4f8f', '#1f968b', '#73d055', '#fde724']
    cmap = LinearSegmentedColormap.from_list('custom', colors, N=100)
    
    # Create figure
    plt.figure(figsize=(10, 8))
    
    # Plot heatmap
    sns.heatmap(
        cm_counts,
        annot=True,
        fmt='d',
        cmap=cmap,
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='white',
        annot_kws={'size': 11, 'weight': 'bold'}
    )
    
    # Add title
    n_test = mask.sum()
    acc = (y_true[mask] == y_pred[mask]).mean()
    plt.title(f'{title} (test n={n_test}, accuracy={acc:.3f})', 
              fontsize=14, fontweight='bold', pad=20)
    
    # Labels
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    
    # Save
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {save_path}")
    plt.close()
    
    return cm_counts, cm_norm


# ============================================================================
# GENERATE CONFUSION MATRICES
# ============================================================================
print("\n" + "="*70)
print("GENERATING CONFUSION MATRICES")
print("="*70)

# Age
print("\n1. Age Group:")
age_cm_counts, age_cm_norm = plot_confusion_matrix(
    merged["age_norm_true"],
    merged["age_norm_pred"],
    age_mask,
    "Age Group Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_age_group.png")
)
if age_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(age_cm_norm.round(3))
    age_cm_norm.to_csv("/data/baliu/python_code/data/cm_age_group.csv")

# Gender
print("\n2. Gender:")
gender_cm_counts, gender_cm_norm = plot_confusion_matrix(
    merged["gender_norm_true"],
    merged["gender_norm_pred"],
    gender_mask,
    "Gender Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_gender.png")
)
if gender_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(gender_cm_norm.round(3))
    gender_cm_norm.to_csv("/data/baliu/python_code/data/cm_gender.csv")

# Household Size
print("\n3. Household Size:")
household_cm_counts, household_cm_norm = plot_confusion_matrix(
    merged["household_norm_true"],
    merged["household_norm_pred"],
    household_mask,
    "Household Size Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_household_size.png")
)
if household_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(household_cm_norm.round(3))
    household_cm_norm.to_csv("/data/baliu/python_code/data/cm_household_size.csv")

# Income Level
print("\n4. Income Level:")
income_cm_counts, income_cm_norm = plot_confusion_matrix(
    merged["income_norm_true"],
    merged["income_norm_pred"],
    income_mask,
    "Income Level Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_income_level.png")
)
if income_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(income_cm_norm.round(3))
    income_cm_norm.to_csv("/data/baliu/python_code/data/cm_income_level.csv")

# ============================================================================
# SAVE MERGED DATA
# ============================================================================
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

merged.to_csv("/data/baliu/python_code/data/merged_evaluation_results.csv", index=False)
print("✓ Merged results saved")

# Save filtered subsets
if income_mask.sum() > 0:
    merged[income_mask].to_csv("/data/baliu/python_code/data/income_evaluation_subset.csv", index=False)
    print("✓ Income subset saved")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("📊 FINAL EVALUATION SUMMARY")
print("="*70)
print(f"\nTotal users: {len(merged)}")
print(f"\n{'Metric':<20} {'Accuracy':<12} {'Samples':<10} {'Coverage'}")
print("-" * 70)
print(f"{'Age Group':<20} {age_acc:>6.1%}        {age_mask.sum():>6d}     {age_mask.sum()/len(merged):>6.1%}")
print(f"{'Gender':<20} {gender_acc:>6.1%}        {gender_mask.sum():>6d}     {gender_mask.sum()/len(merged):>6.1%}")
print(f"{'Household Size':<20} {household_acc:>6.1%}        {household_mask.sum():>6d}     {household_mask.sum()/len(merged):>6.1%}")
print(f"{'Income Level':<20} {income_acc:>6.1%}        {income_mask.sum():>6d}     {income_mask.sum()/len(merged):>6.1%}")

print("\n" + "="*70)
print("✅ EVALUATION COMPLETED")
print("="*70)
print("\nGenerated files:")
print("  📊 cm_age_group.png & .csv")
print("  📊 cm_gender.png & .csv")
print("  📊 cm_household_size.png & .csv")
print("  📊 cm_income_level.png & .csv")
print("  📄 merged_evaluation_results.csv")
print("  📄 income_evaluation_subset.csv")

/tmp/ipykernel_369649/2909887862.py:17: DtypeWarning: Columns (0: postcode_home, 1: education, 2: main_employment, 3: income, 4: household_size, 5: citizen_1, 6: citizen_2, 7: intro_survey_started_at, 8: intro_survey_finished_at, 9: intro_survey_finished, 10: work_status_employed, 11: work_status_self_employed, 12: work_status_unemployed, 13: work_status_apprentice, 14: work_status_student, 15: work_status_retired, 16: work_status_other, 17: own_vehicles_car, 18: own_vehicles_motorbike, 19: own_vehicles_bicycle, 20: car_fuel, 21: car_year, 22: car_size, 23: bike_type_regular, 24: bike_type_ebike_25, 25: bike_type_ebike_45, 26: pt_pass_ga, 27: pt_pass_half_fare, 28: pt_pass_regional_pass, 29: pt_pass_track_7, 30: pt_pass_other, 31: pt_pass_no_pass, 32: freq_cardriver_own_car, 33: freq_cardriver_shared_car, 34: freq_carpass_car_in_hh, 35: freq_carpass_car_pooling, 36: freq_carpass_taxi, 37: freq_carpass_app_based, 38: freq_pt_train, 39: freq_pt_local_pt, 40: freq_bike_own_bike, 41: freq_

HOUSEHOLD INCOME PREDICTION EVALUATION
Ground truth: 90909 rows
Predictions:  2038 rows

NORMALIZING DATA
✓ Normalization complete

📊 Ground Truth Distributions:
Age:       {'18-25': 35836, '36-45': 27024}
Gender:    {'male': 46478, 'female': 44431}
Household: {'2': 7144, '4': 4944, '3': 4320, '1': 3419}
Income:    {'4001-8000': 6454, '8001-12000': 5285, '<4000': 2618, '12001-16000': 2575, '>16000': 1700}

📊 Prediction Distributions:
Age:       {'26-35': 1315, '36-45': 457, '18-25': 245}
Gender:    {'female': 1256, 'male': 418, 'other': 343}
Household: {'4': 1566, '3': 449, '2': 2, '1': 1}
Income:    {'4001-8000': 1205, '8001-12000': 810, '12001-16000': 2, '>16000': 1}

MERGING DATA
✓ Merged 1635 users

🔍 Sample of merged data (first 5 rows):
  user_id age_norm_true age_norm_pred gender_norm_true gender_norm_pred
0   AAGAF         18-25         26-35           female           female
1   AAINS         18-25         26-35             male           female
2   AAQME         18-25        

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path   = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]].copy()
pred = pred[["user_id", "age_group", "gender", "household_income_level"]].copy()

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(gt, pred, on="user_id", how="inner", suffixes=("_true", "_pred"))
print(f"Merged {len(merged)} users")
print(merged.head(5))

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
age_mask = merged["age_group_true"].notna() & merged["age_group_pred"].notna()

# If you don't evaluate household size right now, set hh_mask to all-False safely:
hh_mask = pd.Series(False, index=merged.index)

# Income: evaluate where both exist, and GT is not "prefer not to say"
income_mask = (
    merged["income_level_true"].notna()
    & merged["household_income_level"].notna()
    & (merged["income_level_true"].astype(str).str.lower() != "prefer not to say")
)

print("\nEvaluation sample sizes:")
print(f"Age:        {age_mask.sum()}")
print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# --------------------------------------
age_acc = (
    merged.loc[age_mask, "age_group_true"]
    == merged.loc[age_mask, "age_group_pred"]
).mean() if age_mask.any() else np.nan

income_acc = (
    merged.loc[income_mask, "income_level_true"]
    == merged.loc[income_mask, "household_income_level"]
).mean() if income_mask.any() else np.nan

# --------------------------------------
# Majority-class baselines (TRUE labels only)
# --------------------------------------
def majority_baseline(y_true: pd.Series) -> float:
    y_true = y_true.dropna()
    if len(y_true) == 0:
        return np.nan
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(merged.loc[age_mask, "age_group_true"])
income_baseline = majority_baseline(merged.loc[income_mask, "income_level_true"])

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
print(f"Age group accuracy:     {age_acc:.3f} (baseline {age_baseline:.3f})")
print(f"Income level accuracy:  {income_acc:.3f} (baseline {income_baseline:.3f})")

# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path("/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv")
merged.loc[income_mask].to_csv(filtered_income_path, index=False, encoding="utf-8")

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# Confusion matrices
# --------------------------------------
# Age confusion matrix (row-normalized)
if age_mask.any():
    age_cm = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        normalize="index"
    )
    age_cm_path = Path("/data/baliu/python_code/data/age_confusion_matrix_v5_final.csv")
    age_cm.to_csv(age_cm_path)
    print(f"Age confusion matrix saved to: {age_cm_path}")

# Income confusion matrix (counts + row-normalized)
if income_mask.any():
    income_cm_counts = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"]
    )
    print("\nIncome confusion matrix (counts):")
    print(income_cm_counts)

    income_cm_norm = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"],
        normalize="index"
    )
    print("\nIncome confusion matrix (row-normalized):")
    print(income_cm_norm)

/tmp/ipykernel_278062/1828059493.py:14: DtypeWarning: Columns (0: postcode_home, 1: education, 2: main_employment, 3: income, 4: household_size, 5: citizen_1, 6: citizen_2, 7: intro_survey_started_at, 8: intro_survey_finished_at, 9: intro_survey_finished, 10: work_status_employed, 11: work_status_self_employed, 12: work_status_unemployed, 13: work_status_apprentice, 14: work_status_student, 15: work_status_retired, 16: work_status_other, 17: own_vehicles_car, 18: own_vehicles_motorbike, 19: own_vehicles_bicycle, 20: car_fuel, 21: car_year, 22: car_size, 23: bike_type_regular, 24: bike_type_ebike_25, 25: bike_type_ebike_45, 26: pt_pass_ga, 27: pt_pass_half_fare, 28: pt_pass_regional_pass, 29: pt_pass_track_7, 30: pt_pass_other, 31: pt_pass_no_pass, 32: freq_cardriver_own_car, 33: freq_cardriver_shared_car, 34: freq_carpass_car_in_hh, 35: freq_carpass_car_pooling, 36: freq_carpass_taxi, 37: freq_carpass_app_based, 38: freq_pt_train, 39: freq_pt_local_pt, 40: freq_bike_own_bike, 41: freq_

Loaded ground truth (90909 rows)
Loaded predictions  (2062 rows)


KeyError: "['age_group', 'gender'] not in index"